# File này dùng để tiền xử lý schema

## Pipeline bao gồm
- Thu thập table schema của database
- Tạo metadata cho từng table schema sử dụng LLM
- Lưu vào file .txt riêng biệt

In [1]:
import os
import re
import torch
from langchain_ollama import OllamaLLM
from transformers import AutoTokenizer, AutoModelForCausalLM

d:\python-workspace\NL2SQL-chat\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Thu thập table schema của database

In [2]:
schema_folder_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), 'schema_data')
schema_path = schema_folder_path + "\\table_schema.txt"
schema_path

'd:\\python-workspace\\NL2SQL-chat\\schema_data\\table_schema.txt'

### Tạo metadata cho từng table schema

In [3]:
# Đọc dữ liệu schema
with open(schema_path, mode='r', encoding='utf-8') as f:
    schema_text = f.read()

# Lọc các tabel schema
schema_lst = re.findall(r"CREATE TABLE.*?\);", schema_text, flags=re.DOTALL)

schema_lst

['CREATE TABLE Users (\n    user_id SERIAL PRIMARY KEY,\n    full_name VARCHAR(100) NOT NULL,\n    email VARCHAR(100) UNIQUE NOT NULL,\n    password_hash VARCHAR(255) NOT NULL,\n    phone VARCHAR(20),\n    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n);',
 'CREATE TABLE Categories (\n    category_id SERIAL PRIMARY KEY,\n    category_name VARCHAR(100) NOT NULL,\n    description TEXT\n);',
 'CREATE TABLE Products (\n    product_id SERIAL PRIMARY KEY,\n    product_name VARCHAR(255) NOT NULL,\n    price DECIMAL(10, 2) NOT NULL,\n    stock_quantity INT NOT NULL DEFAULT 0,\n    description TEXT,\n    category_id INT,\n    FOREIGN KEY (category_id) REFERENCES Categories(category_id) ON DELETE SET NULL\n);',
 'CREATE TABLE Shopping_Cart (\n    cart_id SERIAL PRIMARY KEY,\n    user_id INT,\n    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE CASCADE\n);',
 'CREATE TABLE Cart_Items (\n    cart_item_id SERIAL PRIMARY KEY,\n    cart

In [ ]:
system_prompt = """You are an expert Database Administrator. Your task is to analyze SQL DDL (CREATE TABLE) and generate JSON metadata to improve Vector Search for a Text-to-SQL RAG system.

STRICT RULES:
1. Output ONLY a valid JSON object. DO NOT wrap the output in markdown code blocks (e.g., no ```json). DO NOT add any conversational text.
2. The JSON must be exactly 1 level deep (FLAT). All values must be STRINGS. No nested objects or arrays.
3. Write "table_description" and "columns_summary" in English.
4. Generate exactly 3 realistic sample data rows respecting the SQL data types.

EXPECTED JSON SCHEMA:
{
"table_name": "<exact_table_name>",
"table_description": "<A detailed description of the function and business logic of this table in English.>",
"columns_summary": "<col1: meaning | col2: meaning | col3: meaning>",
}

EXAMPLE INPUT:
CREATE TABLE users (
    id INT PRIMARY KEY,
    username VARCHAR(50),
    is_active BOOLEAN
);

EXAMPLE OUTPUT:
{
"table_name": "users",
"table_description": "Stores user account information in the system, including their unique identifier and active status.",
"columns_summary": "id: unique identifier | username: login name | is_active: account status (true/false)"
}"""

In [5]:
llm_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_model.to(device)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def create_batch_inputs(schema_lst, sys_prompt=system_prompt):
    text_lst = []
    for schema in schema_lst:
        messages = [
            {"role":"system", "content":sys_prompt},
            {"role":"user", "content":schema}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        text_lst.append(text)

    model_inputs = tokenizer(
        text_lst,
        padding=True,
        return_tensors="pt"
    ).to(device)
    return model_inputs

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 2606.69it/s]


In [6]:
model_inputs = create_batch_inputs(schema_lst)
print("Đang xử lý batch...")
generated_ids = llm_model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=False  # Dùng cho JSON để kết quả ổn định
)

trimmed_generated_ids = [
    output_ids[len(input_ids):] 
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Giải mã ra danh sách các chuỗi JSON cuối cùng
response_texts = tokenizer.batch_decode(trimmed_generated_ids, skip_special_tokens=True)

# In kết quả
for i, text in enumerate(response_texts):
    print(f"\n--- Kết quả cho bảng {i+1} ---")
    print(text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Đang xử lý batch...

--- Kết quả cho bảng 1 ---
{
"table_name": "Users",
"table_description": "Stores user account information, including their unique identifier, full name, email, hashed password, phone number, and creation timestamp.",
"columns_summary": "user_id: unique identifier | full_name: user's full name | email: user's email address | password_hash: hashed password for security | phone: user's phone number | created_at: timestamp when the user account was created",
"sample_data": "Row 1: 1, 'John Doe', 'john.doe@example.com', 'hashed_password_1', '123-456-7890', '2023-10-01 10:00:00' | Row 2: 2, 'Jane Smith', 'jane.smith@example.com', 'hashed_password_2', '987-654-3210', '2023-10-02 11:00:00' | Row 3: 3, 'Alice Johnson', 'alice.johnson@example.com', 'hashed_password_3', '555-555-5555', '2023-10-03 12:00:00'"
}

--- Kết quả cho bảng 2 ---
{
"table_name": "Categories",
"table_description": "Stores information about different categories used in the system, including a unique ide

### Thêm vào file .txt riêng biệt

In [7]:
metadata_text = '\n'.join(response_texts)
print(metadata_text)

{
"table_name": "Users",
"table_description": "Stores user account information, including their unique identifier, full name, email, hashed password, phone number, and creation timestamp.",
"columns_summary": "user_id: unique identifier | full_name: user's full name | email: user's email address | password_hash: hashed password for security | phone: user's phone number | created_at: timestamp when the user account was created",
"sample_data": "Row 1: 1, 'John Doe', 'john.doe@example.com', 'hashed_password_1', '123-456-7890', '2023-10-01 10:00:00' | Row 2: 2, 'Jane Smith', 'jane.smith@example.com', 'hashed_password_2', '987-654-3210', '2023-10-02 11:00:00' | Row 3: 3, 'Alice Johnson', 'alice.johnson@example.com', 'hashed_password_3', '555-555-5555', '2023-10-03 12:00:00'"
}
{
"table_name": "Categories",
"table_description": "Stores information about different categories used in the system, including a unique identifier, name, and optional description.",
"columns_summary": "category_id: 

In [8]:
with open(schema_folder_path + "\\metadata.txt", mode='w', encoding='utf-8') as f:
    f.write(metadata_text)